# Qari-OCR on Colab — Arabic OCR benchmark

Measures **Qari-OCR** (a Qwen2-VL fine-tune for Arabic) on your own PDF, scoring
the same pages the same way as `tesseract_benchmark.ipynb` so the two tables can
be read side by side.

**Turn the GPU on first:** *Runtime → Change runtime type → T4 GPU → Save.*
Cell 1 stops the notebook if there is none, because this model on CPU is not
slow — it is unusable.

**What the comparison is for.** On this project's fixtures Qari reads Arabic at
roughly a third the word error rate of anything that runs on CPU — **0.063 WER
against 0.172** for `tesseract-best` — and it keeps diacritics and handles inline
English, which the others mangle. It also needs ~5 GB of VRAM and around 30 s on
a real book page against ~2 s. That trade is worth making for a document you care
about and not for a 200-page book you are ingesting in bulk, which is exactly why
the application runs Tesseract inline and treats this as the deliberate path.

This notebook **benchmarks**. To serve Qari over HTTP to the running app, use
`src/ocr/colab/qari_server.ipynb` instead — different job, different notebook.

## 1. GPU check — do this before anything else

In [ ]:
import subprocess, sys

out = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if out.returncode != 0:
    raise SystemExit(
        "No GPU. Runtime -> Change runtime type -> T4 GPU -> Save, then run this cell again.\n"
        "Qari on CPU is not slow, it is unusable: minutes per page, not seconds."
    )
print(out.stdout.split("\n")[8] if len(out.stdout.split("\n")) > 8 else out.stdout[:400])

## 2. Install

Colab preloads its own `transformers`. Upgrading it inside a running kernel
leaves the old module objects imported, and the failure surfaces much later
inside the model load looking like something else entirely — so this cell stops
and tells you to restart rather than letting that happen.

In [ ]:
import sys

if "transformers" in sys.modules:
    raise SystemExit(
        "transformers was already imported in this kernel.\n"
        "Runtime -> Restart session, then run from cell 1. Skipping this is the\n"
        "most common way this notebook fails, and it fails later and confusingly."
    )

!pip -q install --upgrade "transformers>=4.45" accelerate qwen-vl-utils pymupdf pillow
print("installed — no restart needed, transformers had not been imported")

## 3. The PDF

In [ ]:
# --- get a PDF in ---------------------------------------------------------
# Drag a file into the file browser on the left and set PDF_PATH, or run this
# and pick one. For anything large, mount Drive instead -- an upload widget on
# a 25 MB book is slower than Drive and dies on a flaky connection.
from google.colab import files
import os

PDF_PATH = "/content/book.pdf"

if not os.path.exists(PDF_PATH):
    uploaded = files.upload()
    name = next(iter(uploaded))
    os.rename(name, PDF_PATH)

import pymupdf
doc = pymupdf.open(PDF_PATH)
print(f"{doc.page_count} pages")

# Which pages to test. A handful is enough: these engines are seconds per page
# and the numbers stabilise quickly. Pick from the middle -- front matter and
# title pages are not representative of body text.
FIRST, LAST = 40, 47
PAGES = list(range(FIRST, min(LAST + 1, doc.page_count)))
print("testing pages", PAGES)


## 4. Scoring — identical to the Tesseract notebook

In [ ]:
# --- scoring -------------------------------------------------------------
# Both notebooks score the same way so their numbers can sit in one table.
# CER and WER are both reported because on fragmented Arabic they disagree:
# splitting `اليسار` into `ا ليسا ر` changes no letters and destroys every
# word, so CER barely moves while WER collapses -- and WER is the one that
# predicts whether retrieval works.
import unicodedata, re

def normalize(text: str) -> str:
    """NFKC, strip bidi controls, collapse whitespace.

    Presentation forms (U+FB50-FDFF, U+FE70-FEFF) fold to typed letters here,
    so an engine is not punished for emitting a form the pipeline normalises
    away before indexing anyway.
    """
    text = unicodedata.normalize("NFKC", text)
    text = text.translate(dict.fromkeys(map(ord, "\u200e\u200f\u202a\u202b\u202c\u202d\u202e\u2066\u2067\u2068\u2069")))
    return re.sub(r"\s+", " ", text).strip()

def levenshtein(a, b):
    if a == b: return 0
    if not a: return len(b)
    if not b: return len(a)
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ca != cb)))
        prev = cur
    return prev[-1]

def cer(truth, hyp):
    t, h = normalize(truth), normalize(hyp)
    return 1.0 if not t else min(1.0, levenshtein(t, h) / len(t))

def wer(truth, hyp):
    t, h = normalize(truth).split(), normalize(hyp).split()
    return 1.0 if not t else min(1.0, levenshtein(t, h) / len(t))

def space_ratio(text):
    """Intrinsic quality signal for when there is no ground truth.

    Healthy Arabic prose sits around 0.13-0.22. Far below means words fused
    together; far above means they were split mid-word, which is the failure
    this whole exercise is about.
    """
    t = normalize(text)
    return 0.0 if not t else t.count(" ") / len(t)


## 5. Load the model

~5 GB on the first run, cached afterwards for the life of the session.

`min_pixels`/`max_pixels` cap how many vision tokens a page becomes. Qwen2-VL
turns a page into 28×28 patches, so an uncapped 300-dpi render is tens of
thousands of tokens: slow, and beyond the context it was tuned for. Setting the
cap **too low** is worse than too high and fails silently — the page still
"reads", it just returns a handful of characters, which looks like the model
being bad rather than the image being unreadable.

In [ ]:
import torch, time
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor

MODEL = "NAMAA-Space/Qari-OCR-0.2.2.1-VL-2B-Instruct"

t0 = time.perf_counter()
model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL, torch_dtype=torch.bfloat16, device_map="auto",
)
processor = AutoProcessor.from_pretrained(
    MODEL,
    min_pixels=256 * 28 * 28,
    max_pixels=6400 * 28 * 28,   # measured: a lower cap returned ~5 chars/page
)
model.eval()
print(f"loaded in {time.perf_counter()-t0:.0f}s")
print(f"VRAM allocated: {torch.cuda.memory_allocated()/1024**3:.1f} GB")

## 6. Read the pages

In [ ]:
import io, re, time
import pymupdf
from PIL import Image

DPI = 300

def render(page_no, dpi=DPI):
    d = pymupdf.open(PDF_PATH)
    try:
        pix = d[page_no].get_pixmap(dpi=dpi)
        return Image.open(io.BytesIO(pix.tobytes("png"))).convert("RGB")
    finally:
        d.close()

def strip_markup(text: str) -> str:
    """Qari emits light HTML by design.

    Scoring the tags as if they were OCR errors is a mistake I made once and it
    cost the model a factor of four: 0.233 WER became 0.063 once the markup it
    is supposed to produce stopped being counted against it.
    """
    text = re.sub(r"<[^>]+>", " ", text)
    return re.sub(r"\s+", " ", text).strip()

PROMPT = "Below is the image of one page of a document. Extract all text exactly as it appears."

@torch.inference_mode()
def read(page_no):
    image = render(page_no)
    messages = [{"role": "user", "content": [
        {"type": "image", "image": image},
        {"type": "text", "text": PROMPT},
    ]}]
    chat = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[chat], images=[image], return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=2048, do_sample=False)
    trimmed = out[0][len(inputs.input_ids[0]):]
    return processor.decode(trimmed, skip_special_tokens=True)

texts, per_page = {}, {}
for n in PAGES:
    t0 = time.perf_counter()
    texts[n] = strip_markup(read(n))
    per_page[n] = time.perf_counter() - t0
    print(f"  page {n}: {per_page[n]:5.1f}s  {len(texts[n]):5d} chars  space ratio {space_ratio(texts[n]):.3f}")

print(f"\nmean {sum(per_page.values())/len(per_page):.1f} s/page on {torch.cuda.get_device_name(0)}")

## 7. What it read

Check this by eye before trusting any number below it. Diacritics present,
words not split mid-token, inline English intact — those are the things Qari is
supposed to be better at.

In [ ]:
for n in PAGES[:2]:
    print(f"--- page {n} ---")
    print(texts[n][:600])
    print()

## 8. Scores

Supply ground truth for at least one page to get CER and WER. Without it this
reports the intrinsic signal only — useful, but not a comparison.

In [ ]:
TRUTH = {
    # 40: "اليسار حينئذ بديدو ومعناه الهاربة ...",
}

print(f"{'page':>5}  {'chars':>6}  {'s/page':>7}  {'space ratio':>12}")
for n, t in texts.items():
    print(f"{n:>5}  {len(normalize(t)):>6}  {per_page[n]:>7.1f}  {space_ratio(t):>12.3f}")

if TRUTH:
    print(f"\n{'page':>5}  {'CER':>6}  {'WER':>6}")
    for n, truth in TRUTH.items():
        print(f"{n:>5}  {cer(truth, texts[n]):>6.3f}  {wer(truth, texts[n]):>6.3f}")
else:
    print("\nNo ground truth supplied — CER/WER skipped. Paste one page's correct")
    print("text into TRUTH to get them; one careful page beats ten guessed ones.")

## 9. Reading the result against Tesseract

Put this notebook's numbers next to `tesseract_benchmark.ipynb` run on the
**same pages of the same PDF**. What the comparison should tell you:

| | `tesseract-best` | `qari` |
| --- | ---: | ---: |
| WER (this project's fixtures) | 0.172 | **0.063** |
| s/page, real book page | ~2 (fast CPU) · ~5-6 (t3.medium) | ~30 (T4) |
| needs | nothing | ~5 GB VRAM |

Qari is roughly an order of magnitude slower per page and about three times more
accurate. Neither number decides on its own — what decides is how many pages you
have and whether the document is worth re-reading properly.

Two things worth carrying away rather than re-deriving:

- **A GPU hour is not the constraint; the tunnel is.** A Colab session ends after
  ~12 hours and ~90 minutes idle, and a free ngrok hostname changes every restart.
  This is a workbench. Ingestion that depends on it breaks when the session does.
- **Documents leave your machine** — to Google, and through ngrok if you serve it.
  Fine for a published book, worth a thought for anything else. `tesseract-best`
  runs locally and costs 0.17 WER instead of 0.06.